# 01 — PDF Ingestion & English Extraction

**Purpose:** Explore the raw PDF, understand the page structure,
and extract only English-language question pages.



---

## 1. Setup

In [3]:
import sys
sys.path.append('..')   # so we can import from src/ later

import pdfplumber
import re
import json
from pathlib import Path

# Paths
RAW_DIR   = Path('../data/raw')
OUT_DIR   = Path('../data/processed')
OUT_DIR.mkdir(exist_ok=True)

print('✅ Setup done')

✅ Setup done


## 2. Explore a single PDF — check page count & raw text

In [4]:
PDF_PATH = RAW_DIR / 'Mathematics_Basic_430-1-1.pdf'

with pdfplumber.open(PDF_PATH) as pdf:
    print(f'Total pages: {len(pdf.pages)}')
    print()
    # Look at first 5 pages raw
    for i, page in enumerate(pdf.pages[:5]):
        text = page.extract_text() or ''
        print(f'--- PAGE {i+1} (first 200 chars) ---')
        print(text[:200])
        print()

Total pages: 27

--- PAGE 1 (first 200 chars) ---
Series : GE1FH SET~1
>
. - 43 0/1/1
Q.P. Code
Roll No.
- -
-
Candidates must write the Q.P. Code
on the title page of the answer-book.
> NOTE
(I) - (I) Please check that this question paper
contains 2

--- PAGE 2 (first 200 chars) ---
(i) - 38
(ii) - , , ,
(iii) 1 18 (MCQ) 19 20
1
(iv) 21 25 - (VSA) 2
(v) 26 31 - (SA) 3
(vi) 32 35 - (LA) 5
(vii) 36 38 4
2
(viii) - , 2 , 2 ,
2 3
(ix) = ,
(x)
20 (MCQ) , 1 20 1=20
1. a b (HCF) 1 ,
(LC

--- PAGE 3 (first 200 chars) ---
General Instructions :
Read the following instructions very carefully and strictly follow them :
(i) This question paper contains 38 questions. All questions are compulsory.
(ii) This question paper i

--- PAGE 4 (first 200 chars) ---
4. x + = 3 (x 0) ax2 + bx + c = 0
a b + c
(A) 5 (B) 2
(C) 1 (D) 1
5. (3, 5)
(A) 8 (B) 2
(C) 2 (D) 8
6. - ,
(A) 1 : 2 (B) 2 : 1
(C) 1 : 1 (D) : 2
7. - ?
(A) AAA (B) SSS
(C) SAS (D) RHS
8. , P - ?
(A) P

--- PAGE 5 (first 200 chars) ---
4. The equ

## 3. Identify Hindi vs English pages

In [5]:
def is_english_page(text: str) -> bool:
    """Returns True if page contains English question content."""
    english_patterns = [
        r'\d+\.\s+[A-Z][a-z]',
        r'(Find|Prove|If |The |A |An |SECTION|Case Study|Determine|Show that)',
    ]
    hindi_pattern = r'[\u0900-\u097F]'   # Devanagari unicode block

    has_hindi   = bool(re.search(hindi_pattern, text))
    has_english = any(re.search(p, text) for p in english_patterns)

    return has_english and not has_hindi


# Map every page
with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ''
        label = '✅ ENGLISH' if is_english_page(text) else '⛔ SKIP'
        print(f'Page {i+1:2d}: {label}  | {text[:60].strip()}')

Page  1: ✅ ENGLISH  | Series : GE1FH SET~1
>
. - 43 0/1/1
Q.P. Code
Roll No.
- -
-
Page  2: ⛔ SKIP  | (i) - 38
(ii) - , , ,
(iii) 1 18 (MCQ) 19 20
1
(iv) 21 25 -
Page  3: ✅ ENGLISH  | General Instructions :
Read the following instructions very
Page  4: ✅ ENGLISH  | 4. x + = 3 (x 0) ax2 + bx + c = 0
a b + c
(A) 5 (B) 2
(C) 1
Page  5: ✅ ENGLISH  | 4. The equation x + = 3 (x 0) is expressed as a quadratic eq
Page  6: ✅ ENGLISH  | 9. , O PA - OP = 10 cm , AP
(A) 10 cm
(B) 20 cm
(C) 5 cm
(D)
Page  7: ✅ ENGLISH  | 9. In the given figure, PA is a tangent to a circle with cen
Page  8: ⛔ SKIP  | 12. , - ?
(A) x
(B) y
(C) z
(D) a
13.
(A) l
(B) l + a
(C) l
Page  9: ✅ ENGLISH  | 12. In the given figure, which of the following angles repre
Page 10: ⛔ SKIP  | 15. ?
(A)
(B)
(C)
(D)
16. -
- 10 25 25 40 40 55 55 70 70 85
Page 11: ✅ ENGLISH  | 15. For which of the following solids is the lateral/curved
Page 12: ⛔ SKIP  | 19 20
(A) (R)
(A), (B), (C) (D)
(A) (A) (R) (R), (A)
(B) (A)
Page 13: ✅ ENGLISH  | 

## 4. Extract all English pages from one PDF

In [6]:
def extract_english_pages(pdf_path: Path, paper_name: str, year: str) -> list:
    """Extract only English question pages from a CBSE bilingual PDF."""
    results = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text()
            if not text:
                continue
            text = text.strip()
            if is_english_page(text):
                results.append({
                    'page_num':  i + 1,
                    'paper':     paper_name,
                    'year':      year,
                    'text':      text
                })
    return results


pages = extract_english_pages(
    PDF_PATH,
    paper_name='Mathematics_Standard_SET2',
    year='2024'
)

print(f'Extracted {len(pages)} English pages')
print()
for p in pages:
    print(f"  Page {p['page_num']:2d}: {p['text'][:70]}")

Extracted 19 English pages

  Page  1: Series : GE1FH SET~1
>
. - 43 0/1/1
Q.P. Code
Roll No.
- -
-
Candidate
  Page  3: General Instructions :
Read the following instructions very carefully 
  Page  4: 4. x + = 3 (x 0) ax2 + bx + c = 0
a b + c
(A) 5 (B) 2
(C) 1 (D) 1
5. (
  Page  5: 4. The equation x + = 3 (x 0) is expressed as a quadratic equation in 
  Page  6: 9. , O PA - OP = 10 cm , AP
(A) 10 cm
(B) 20 cm
(C) 5 cm
(D) 5 cm
10. 
  Page  7: 9. In the given figure, PA is a tangent to a circle with centre O. If

  Page  9: 12. In the given figure, which of the following angles represents the 
  Page 11: 15. For which of the following solids is the lateral/curved surface ar
  Page 13: Questions number 19 and 20 are Assertion and Reason based questions. T
  Page 14: 22. , PQ RS , POQ ~ SOR.
, OSR ~ OQP, ROQ = 125 ORS = 70 .
OSR OQP
23.
  Page 15: 22. (a) In the given figure, if PQ RS, then prove that POQ ~ SOR.
OR
(
  Page 17: SECTION C
This section has 6 Short Answer (SA) type questio

## 5. Run on all PDFs in data/raw/ and save

In [7]:
# Auto-discover all PDFs in data/raw/
pdf_files = list(RAW_DIR.glob('*.pdf'))
print(f'Found {len(pdf_files)} PDFs: {[f.name for f in pdf_files]}')

all_pages = []
for pdf_path in pdf_files:
    # Derive paper name from filename
    paper_name = pdf_path.stem.replace('-', '_')
    pages = extract_english_pages(pdf_path, paper_name=paper_name, year='2024')
    all_pages.extend(pages)
    print(f'  {pdf_path.name} → {len(pages)} pages')

# Save
out_path = OUT_DIR / 'raw_english_pages.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_pages, f, indent=2, ensure_ascii=False)

print(f'\n✅ Saved {len(all_pages)} pages → {out_path}')

Found 53 PDFs: ['Mathematics_Standard_30-1-3.pdf', 'Mathematics_Standard_30-S-1_Supplementary_v2.pdf', 'Mathematics_Standard_30-2-3.pdf', 'Mathematics_Standard_30-B-S_Blind_Supplementary.pdf', 'Mathematics_Standard_30-5-1.pdf', 'Mathematics_Standard_30-6-1.pdf', 'Mathematics_Basic_430-S-2_Supplementary.pdf', 'Mathematics_Standard_30-6-3.pdf', 'Mathematics_Basic_430-6-1.pdf', 'Mathematics_Basic_430-1-1.pdf', 'Mathematics_Basic_430-5-1.pdf', 'Mathematics_Basic_430-S-2_Supplementary_v2.pdf', 'Mathematics_Standard_30-5-2.pdf', 'Mathematics_Standard_30-2-2.pdf', 'Mathematics_Basic_430-3-3.pdf', 'Mathematics_Standard_30-S-2_Supplementary_v2.pdf', 'Mathematics_Standard_30-S-3_Supplementary.pdf', 'Mathematics_Basic_430-B-S_Blind_Supplementary.pdf', 'Mathematics_Basic_430-5-2.pdf', 'Mathematics_Basic_430-2-2.pdf', 'Mathematics_Basic_430-4-1.pdf', 'Mathematics_Basic_430-2-3.pdf', 'Mathematics_Standard_30-3-1.pdf', 'Mathematics_Basic_430-5-3.pdf', 'Mathematics_Basic_430-3-2.pdf', 'Mathematics_Sta